In [ ]:
import cv2
import keras
import threading
import pygame

# Khởi tạo pygame mixer
pygame.mixer.init()

# Biến trạng thái âm thanh
alert_playing = False
alert_lock = threading.Lock()

# Hàm phát âm thanh trong luồng riêng
def play_alert():
    global alert_playing
    with alert_lock:
        if not alert_playing:
            def _play():
                pygame.mixer.music.load("alert.mp3")
                pygame.mixer.music.play(-1)  # Lặp vô hạn
            threading.Thread(target=_play, daemon=True).start()
            alert_playing = True

# Hàm dừng âm thanh
def stop_alert():
    global alert_playing
    with alert_lock:
        if alert_playing:
            pygame.mixer.music.stop()
            alert_playing = False

# Load mô hình và cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
model = keras.models.load_model('drowsiness_model.keras')

# Biến đếm
drowsy_counter = 0
active_counter = 0
THRESHOLD = 12
RESET_THRESHOLD = 4  # Tắt âm thanh sau khi tỉnh táo 4 khung hình

# Webcam
cap = cv2.VideoCapture(0)
print("🎥 Bắt đầu phát hiện buồn ngủ. Nhấn Q để thoát.")

def predict_drowsiness(face_roi, model):
    face_resized = cv2.resize(face_roi, (224, 224))
    face_normalized = face_resized / 255.0
    face_input = face_normalized.reshape(1, 224, 224, 3)
    prediction = model.predict(face_input, verbose=0)[0]

    label = 'drowsy' if prediction[0] > 0.5 else 'active'
    prob = prediction[0] if label == 'drowsy' else 1 - prediction[0]
    return label, prob

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    detected_label = None

    for (x, y, w, h) in faces:
        face_roi = frame[y:y + h, x:x + w]
        label, prob = predict_drowsiness(face_roi, model)
        detected_label = label

        if label == 'drowsy':
            drowsy_counter += 1
            active_counter = 0
            if drowsy_counter >= THRESHOLD:
                play_alert()
        else:
            active_counter += 1
            if active_counter >= RESET_THRESHOLD:
                drowsy_counter = 0
                stop_alert()

        color = (0, 255, 0) if label == 'active' else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        cv2.putText(frame, f"{label} ({prob:.2f})", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.imshow('Drowsiness Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Dọn tài nguyên
cap.release()
cv2.destroyAllWindows()
stop_alert()


error: ALSA: Couldn't open audio device: No such file or directory